In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import pandas as pd
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go


In [2]:
# ======== HELPER FUNCTIONS ========
def get_file_path(data_type):
    """Get file path based on data type"""
    if data_type.lower() == "sale":
        return r"E:\pgpt\datas\sales\sales_from_2024.csv"
    elif data_type.lower() == "rent":
        return r"E:\pgpt\datas\rents\rents_from_2024.csv"\

def load_and_prepare_data(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    df['price_per_square'] = df.apply(
        lambda row: row['price'] / row['area'] if row['area'] != 0 else None,
        axis=1
    )
    return df

def filter_data(df, column):
    p1 = df[column].quantile(0.01)
    p99 = df[column].quantile(0.99)
    return df[(df[column] >= p1) & (df[column] <= p99)]

def create_subplot_figure(df, property_types, target_column, title):
    """Create subplot figure for property types"""
    max_count = 0
    hist_data = {}
    
    for ptype in property_types:
        counts, bin_edges = np.histogram(
            df[df["property_type"] == ptype][target_column], 
            bins=500
        )
        counts = counts + 1  
        hist_data[ptype] = (counts, bin_edges)
        max_count = max(max_count, counts.max())
    
    fig = make_subplots(
        rows=len(property_types),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02
    )
    
    for i, ptype in enumerate(property_types, start=1):
        counts, bin_edges = hist_data[ptype]
        bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
        
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=counts,
                marker_line_color='black',
                marker_line_width=0.5,
                opacity=0.6,
                name=ptype
            ),
            row=i,
            col=1
        )
        
        fig.update_yaxes(
            type="log",
            range=[0, np.log10(max_count)],
            showticklabels=(i == 1),
            row=i,
            col=1
        )
    
    fig.update_layout(
        height=250 * len(property_types),
        title=title,
        bargap=0.05,
        legend_title="Property Type",
        margin=dict(l=0, r=0, t=40, b=0)
    )
    return fig
# ======== PLOTTING FUNCTIONS ========
def plot_basic_histogram(data_type, metric):
    """Plot basic histogram for selected metric"""
    file_path = get_file_path(data_type)
    df = load_and_prepare_data(file_path)
    df_filtered = filter_data(df, metric)
    
    # Set labels and titles based on metric
    if metric == "price":
        x_label = "Price"
        title_metric = "Property Prices"
        nbins = 1000 if data_type == "rent" else 500
    else:
        x_label = "Price per Square"
        title_metric = "Price per Square"
        nbins = 500
    
    fig = px.histogram(
        df_filtered,
        x=metric,
        nbins=nbins,
        title=f"Histogram of {title_metric} ({data_type.capitalize()}, 1st to 99th Percentile)",
        labels={metric: x_label, "count": "Number of Properties"},
        log_x=True
    )
    fig.update_traces(marker_line_color='black', marker_line_width=1)
    fig.update_layout(bargap=0.05)
    fig.show()

def plot_log_histogram(data_type, metric):
    """Plot log-scale histogram using matplotlib/seaborn"""
    file_path = get_file_path(data_type)
    df = load_and_prepare_data(file_path)
    df_filtered = filter_data(df, metric)
    
    # Set labels and titles based on metric
    if metric == "price":
        x_label = "Price (log scale)"
        title_metric = "Property Prices"
        bins = 120
    else:
        x_label = "Price per Square (log scale)"
        title_metric = "Property Price per Square"
        bins = 100
    
    plt.figure(figsize=(8, 6))
    sns.histplot(
        data=df_filtered,
        x=metric,
        bins=bins,
        edgecolor="black",
        kde=True,
        log_scale=(True, False),
    )
    plt.xlabel(x_label)
    plt.ylabel("Number of Properties")
    plt.title(f"Histogram of {title_metric} ({data_type.capitalize()}, Log X-Axis)")
    plt.grid(axis='y', alpha=0.3)
    plt.show()

def plot_property_type_distribution(data_type, metric):
    """Plot property distributions by type for selected metric"""
    file_path = get_file_path(data_type)
    df = load_and_prepare_data(file_path)
    df_filtered = filter_data(df, metric)
    
    # Set labels and titles based on metric
    if metric == "price":
        x_label = "Price"
        title_metric = "Property Price"
    else:
        x_label = "Price per Square"
        title_metric = "Price per Square"
    
    fig = px.histogram(
        df_filtered,
        x=metric,
        nbins=500,
        color="property_type",
        barmode="overlay",
        title=f"{title_metric} Distribution by Property Type ({data_type.capitalize()})",
        labels={metric: x_label, "count": "Number of Properties"}
    )
    fig.update_traces(marker_line_color='black', marker_line_width=0.5, opacity=0.6)
    fig.update_layout(bargap=0.05, legend_title="Property Type")
    fig.show()

def plot_detailed_property_type(data_type, metric):
    """Generate detailed property type analysis for selected metric"""
    file_path = get_file_path(data_type)
    df = load_and_prepare_data(file_path)
    df_filtered = filter_data(df, metric)
    
    # Set titles based on metric
    if metric == "price":
        title_metric = "Property Price"
    else:
        title_metric = "Price per Square"
    
    # Get property types sorted by count
    type_counts = df_filtered["property_type"].value_counts()
    property_types = type_counts.index.tolist()
    
    fig = create_subplot_figure(
        df_filtered, 
        property_types, 
        metric,
        f"{title_metric} Distribution by Property Type ({data_type.capitalize()}, Sorted by Count)"
    )
    fig.show()
# ======== MAIN PROGRAM ========
if __name__ == "__main__":
    # Step 1: Ask for data type (sale/rent)
    data_choice = input("Do you want to process 'sale' or 'rent' data? ").strip().lower()
    if data_choice not in ['sale', 'rent']:
        print("Invalid choice! Please enter either 'sale' or 'rent'.")
        exit()
    
    # Step 2: Ask for metric (price/price_per_square)
    metric_choice = input("Do you want to analyze 'price' or 'price_per_square'? ").strip().lower()
    if metric_choice not in ['price', 'price_per_square']:
        print("Invalid choice! Please enter either 'price' or 'price_per_square'.")
        exit()
    
    # Step 3: Show visualization menu
    print("\nAvailable visualizations:")
    print("1. Basic histogram (Plotly)")
    print("2. Log-scale histogram (Matplotlib/Seaborn)")
    print("3. Property type distribution (overlaid histograms)")
    print("4. Detailed property type analysis (subplots per type)")
    
    # Get plot choice
    try:
        plot_choice = int(input("Enter the number of the visualization you want to see: "))
    except ValueError:
        print("Please enter a valid number (1-4)")
        exit()
    
    # Execute chosen plot
    if plot_choice == 1:
        plot_basic_histogram(data_choice, metric_choice)
    elif plot_choice == 2:
        plot_log_histogram(data_choice, metric_choice)
    elif plot_choice == 3:
        plot_property_type_distribution(data_choice, metric_choice)
    elif plot_choice == 4:
        plot_detailed_property_type(data_choice, metric_choice)
    else:
        print("Invalid choice! Please select a number between 1 and 4.")


Available visualizations:
1. Basic histogram (Plotly)
2. Log-scale histogram (Matplotlib/Seaborn)
3. Property type distribution (overlaid histograms)
4. Detailed property type analysis (subplots per type)
